[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/g2_rolling/g2_rolling.ipynb)

# Octonionic Rolling

This example plans the rolling of a marked ball over a fixed sphere three times its radius, rolling without slipping or twisting. A configuration is where the ball touches the sphere and how it is turned, and the question is which rolls lead from one configuration to another. At this ratio of radii the configurations are the null split octonions, up to scale, and rolling is read off from their product: two configurations one roll apart multiply to zero.

A split octonion is modelled as a multivector of space whose even part, and odd part divided by the volume, are two even multivectors, coupled by the product `multiply`; a configuration packs the contact direction into the one and the contact turned by the ball's attitude into the other. Multiplication by the start, `multiply(start, State)`, is a map whose kernel holds every configuration one roll away, so a route through waypoints comes from a least-squares projection and one more product, not from a search. Maps that preserve the product, built from left and right multiplications, carry routes to routes: they form the exceptional group split G₂.

In the notation of Baez and Huerta the configurations form the projectivized null cone of the imaginary split octonions $\mathrm{Im}\,\mathbb{O}'$, two configurations are one roll apart when $xy = 0$, and the symmetries form $G_2' = \mathrm{Aut}(\mathbb{O}')$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.animation import show_animation
from examples.math.g2_rolling import render

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
Even = ga.gatype.even()
Octonion = ga.gatype.full()
State = ga.gatype.nonscalar()
StateMap = ga.gatype((State, State))
# The volume packs a second even multivector into the odd grades.
volume = mv.xyz                                                       # Trivector

## 1. Configurations as split octonions

A configuration packs a contact direction and an attitude rotor into a null split octonion. Even and odd grades hold its two parts; the multiplication below couples them. The contact direction is projective, so antipodal representatives encode the same configuration. Choosing a hemisphere gives the continuous sphere lift drawn here. At the 1:3 radius ratio, each great-circle roll becomes a straight projective line of these states.

In [ ]:
def multiply(first: Octonion, second: Octonion) -> Octonion:
    """Split-octonion multiplication, packing two even multivectors into even and odd grades."""
    first_even = first.restrict.even()                         # Even
    second_even = second.restrict.even()                       # Even
    first_partner = first.restrict.odd() * volume.inverse()     # Even
    second_partner = second.restrict.odd() * volume.inverse()   # Even
    # The partners enter in the opposite order; reversal supplies their conjugation.
    even = first_even * second_even + second_partner * first_partner.reverse()  # Even
    partner = first_even.reverse() * second_partner + second_even * first_partner  # Even
    return even + partner * volume

def pairing(first: Octonion, second: Octonion) -> Scalar:
    """The split norm's bilinear form: contact part minus attitude part."""
    return first.clifford_conjugate().scalar_product(second)

def encode(contact: Vector, attitude: Rotor) -> State:
    """A unit contact direction and ball attitude as one null split octonion."""
    contact_plane = contact * volume                           # Bivector
    return contact_plane + contact_plane * attitude * volume

def normalize(state: State) -> State:
    """Choose the null representative with unit contact direction."""
    return state / state.restrict[2].norm()

def decode(state: State) -> tuple[Vector, Rotor]:
    """Recover the contact direction and ball attitude from a null representative."""
    state = normalize(state)                                   # State
    contact_plane = state.restrict[2]                          # Bivector
    partner = state.restrict.odd() / volume                    # Even
    return contact_plane / volume, contact_plane.inverse() * partner

def lift(state: State, hemisphere: Vector) -> State:
    """Choose the representative whose contact lies in the specified open hemisphere."""
    direction = state.restrict[2] / volume                     # Vector
    return normalize(state * direction.scalar_product(hemisphere))

def roll(start: State, finish: State, fractions: np.ndarray) -> State:
    """Sample the projective line between two configurations whose product vanishes."""
    return normalize(start * (1 - fractions) + finish * fractions)

## 2. The waypoint from a product

Two configurations whose split product vanishes are connected by one great-circle roll. If their split pairing vanishes but their product does not, the product is the unique intermediate configuration for two rolls.

In [ ]:
attitude_angle = -2.4
sphere_radius = 3
ball_radius = 1
n_samples = 120
n_steps = 36
n_hold_frames = 12
duration_ms = 45
elevation = 35
azimuth = -25
size = 520
samples = np.linspace(0, 1, n_samples)
fractions = np.linspace(0, 1, n_steps)

start_contact = mv.z                                                   # Vector
start_attitude = mv.rotor()                                            # Rotor
target_contact = mv.x                                                  # Vector
target_attitude = (mv.xy * (-attitude_angle / 2)).exp()                   # Rotor
preferred_contact = mv.x                                               # Vector

In [ ]:
start = encode(start_contact, start_attitude)                          # State
finish = encode(target_contact, target_attitude)                        # State
# These endpoints have zero split pairing. Their product supplies the waypoint.
waypoint = lift(multiply(start, finish).cast(State), preferred_contact)  # State
stops = stack([start, waypoint, finish])                                # [stops] State

def paths(stops: State, fractions: np.ndarray) -> tuple[Vector, ...]:
    """Sample each contact arc for the fixed route drawing."""
    states = roll(stops[:-1, None], stops[1:, None], fractions)    # [legs, samples] State
    contacts, _ = decode(states)                                 # [legs, samples] Vector, Rotor
    return tuple(contacts)

def rolling(stops: State, fractions: np.ndarray, n_hold_frames: int) -> Iterator[tuple[Vector, Rotor]]:
    """Yield the contact and attitude along each algebraically constructed roll."""
    for _ in range(n_hold_frames):
        yield decode(stops[0])
    for first, second in zip(stops[:-1], stops[1:]):
        for fraction in fractions:
            yield decode(first * (1 - fraction) + second * fraction)
    for _ in range(n_hold_frames):
        yield decode(stops[-1])

show_animation(render.frames(paths(stops, samples), rolling(stops, fractions, n_hold_frames),
                             sphere_radius, ball_radius, elevation, azimuth, size),
               "g2_two_rolls", duration_ms)

The blue and orange arcs are the two rolls. The paint stays fixed to the small ball, making its attitude visible. Each arc is a line in the configuration geometry: its decoded motion rolls without slipping or twisting.

## 3. The route from a kernel

Returning to the same contact point with a different attitude requires a longer route. The kernel of multiplication by the start contains its admissible first waypoints. Requiring zero pairing with the finish leaves configurations that can reach it in two more rolls.

In [ ]:
target_contact = start_contact                                        # Vector
contact_preference = preferred_contact * volume                        # Bivector

In [ ]:
finish = encode(target_contact, target_attitude)                        # State
# The least-squares round trip keeps the part detected by multiplication.
# Its complement is the kernel, containing the admissible first rolls.
action = multiply(start, State)                                        # Octonion <- State
reachable = State - action.lstsq(action)                                # State <- State

# Remove the goal-pairing component of the preference, then apply reachability.
remaining = pairing(finish, reachable)                                 # Scalar <- State
first = reachable(contact_preference - remaining.lstsq(remaining(contact_preference)))  # State
second = multiply(first, finish).cast(State)                            # State
first = lift(first, preferred_contact)                                  # State
second = lift(second, preferred_contact)                                # State
stops = stack([start, first, second, finish])                            # [stops] State

show_animation(render.frames(paths(stops, samples), rolling(stops, fractions, n_hold_frames),
                             sphere_radius, ball_radius, elevation, azimuth, size),
               "g2_three_rolls", duration_ms)

The ball follows the blue, orange and purple arcs and returns to its starting contact with the requested attitude. The route comes from the multiplication kernel and one further product; no search over rolling trajectories is needed.

## 4. A symmetry of whole routes

The invertible maps preserving split-octonion multiplication form split G₂. Such a map preserves every vanishing neighboring product, so it carries an admissible route to another one. Left and right multiplication actions construct an infinitesimal symmetry; the choice below fixes the start while changing the rest of the route.

In [ ]:
symmetry_amount = 0.12
panel_size = 370

In [ ]:
def derivation(first: Octonion, second: Octonion) -> StateMap:
    """An infinitesimal product-preserving map, assembled from multiplication actions."""
    left_first = multiply(first, Octonion)                     # Octonion <- Octonion
    left_second = multiply(second, Octonion)                   # Octonion <- Octonion
    right_first = multiply(Octonion, first)                    # Octonion <- Octonion
    right_second = multiply(Octonion, second)                  # Octonion <- Octonion
    action = (left_first(left_second) - left_second(left_first)
              + left_first(right_second) - right_second(left_first)
              + right_first(right_second) - right_second(right_first))
    return action(State).cast(State)

# This generator's third composition vanishes, so its exponential terminates here.
generator = derivation(start, mv.x)                                    # State <- State
symmetry = State + symmetry_amount * generator + symmetry_amount**2 / 2 * generator(generator)  # State <- State
transformed_stops = symmetry(stops)                                    # [stops] State

show_animation(render.comparison_frames(
    paths(stops, samples), paths(transformed_stops, samples),
    rolling(stops, fractions, n_hold_frames), rolling(transformed_stops, fractions, n_hold_frames),
    sphere_radius, ball_radius, elevation, azimuth, panel_size),
    "g2_route_symmetry", duration_ms)

In [ ]:
# checks
# Neighbouring stops of the route, and of its image under the symmetry, multiply to zero: each pair is one
# roll apart.
np.testing.assert_allclose(multiply(stops[:-1], stops[1:]).kernel, 0, atol=1e-10)
np.testing.assert_allclose(multiply(transformed_stops[:-1], transformed_stops[1:]).kernel, 0, atol=1e-10)

The left route and its right-hand image begin at the same configuration. The symmetry changes the route and its destination, preserving rolling. It acts on configurations rather than rigidly rotating the two spheres.

The construction follows Baez and Huerta's [*G₂ and the Rolling Ball*](https://arxiv.org/html/1205.2447v4), particularly Proposition 7 and Theorem 10. Their global geometry uses projective contact directions and spinor attitudes; each displayed route is a continuous lift to physical rolling on the sphere.